## Preparing the data

### here is the dataset link we used in this notebook

https://universe.roboflow.com/arpitha-m/100-ycs08

In [1]:
!scp '/content/100.v1i.coco.zip'

usage: scp [-346ABCOpqRrsTv] [-c cipher] [-D sftp_server_path] [-F ssh_config]
           [-i identity_file] [-J destination] [-l limit]
           [-o ssh_option] [-P port] [-S program] source ... target


## dataset used in this notebook is not same as in the video, so note and cant verify metrics

In [2]:
!unzip '/content/100.v1i.coco.zip' -d '/content/dataset'

Archive:  /content/100.v1i.coco.zip
 extracting: /content/dataset/README.dataset.txt  
 extracting: /content/dataset/README.roboflow.txt  
   creating: /content/dataset/test/
 extracting: /content/dataset/test/_annotations.coco.json  
 extracting: /content/dataset/test/img_1_jpg.rf.cd1e49316285d3afaa529f99624a1829.jpg  
 extracting: /content/dataset/test/img_24_jpg.rf.f81ff52ee17a0b2024e660756e267754.jpg  
 extracting: /content/dataset/test/img_31_jpg.rf.1398c1cc92a777622625fc34add2eb09.jpg  
 extracting: /content/dataset/test/img_51_jpg.rf.1a17aaffb67efb7c962e189888778262.jpg  
 extracting: /content/dataset/test/img_61_jpg.rf.dafedaac037303207c0c5634e59665f0.jpg  
 extracting: /content/dataset/test/img_74_jpg.rf.7eff4e4c2b0e55d0f012e0d7c291f9f8.jpg  
 extracting: /content/dataset/test/img_76_jpg.rf.22a39157eecf3e5cec12526f7109a0e3.jpg  
 extracting: /content/dataset/test/img_79_jpg.rf.bf128801f012af29ac7d19e10b5962c9.jpg  
 extracting: /content/dataset/test/img_80_jpg.rf.68fe4fdc2e217

## Installing the dependencies

In [3]:
!apt install tesseract-ocr
!apt install libtesseract-dev

!pip install pytesseract
!pip install Pillow
!pip install easyocr
!pip install boto3

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
tesseract-ocr is already the newest version (4.1.1-2.1build1).
0 upgraded, 0 newly installed, 0 to remove and 1 not upgraded.
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libarchive-dev libleptonica-dev
The following NEW packages will be installed:
  libarchive-dev libleptonica-dev libtesseract-dev
0 upgraded, 3 newly installed, 0 to remove and 1 not upgraded.
Need to get 3,743 kB of archives.
After this operation, 16.0 MB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 libarchive-dev amd64 3.6.0-1ubuntu1.5 [581 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libleptonica-dev amd64 1.82.0-3build1 [1,562 kB]
Get:3 http://archive.ubuntu.com/ubuntu jammy/universe amd64 libtesseract-dev amd64 4.1.1-2.1build1 [1,600 kB]


# Detection using
### 1. Pytesseract

In [5]:
import pytesseract
from PIL import Image

image_path = '/content/dataset/train/img_75_jpg.rf.3ce459042d7664fe4cf4fed6f0854c0d.jpg'
text = pytesseract.image_to_string(Image.open(image_path))
print(text)

————

 

WHIL
YOU
WALT

SHOE. ~~
“REPAIRS

AVAILABLE

 



## 2. easyocr

In [6]:
from easyocr import Reader
text = ''
reader = Reader(['en'])
results = reader.readtext(Image.open(image_path))

Progress: |██████████████████████████████████████████████████| 100.0% Complete

Progress: |██████████████████████████████████████████████████| 100.0% Complete

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


In [8]:
# the results contain different values
# but the the text of each line is present in the 1st index of each line
# so we follow the below procedure
for result in results:
  text = text+result[1]+' '

# reversing the added text for visualising
print(text)

WHILE YOU WA LT SHOE REPA !RS AVA ILABLE 


## 3. AWS Text extractor

- involved
 so many steps involved in this process including access key tokens,  but asking for pricing

-  Generally jacob similarity is used to compare the similarity between the two sentences

- based on that score we can say which is better

- for the above we should have a dataset which includes image name same as the text included within the image.

- we dont have such dataset so we are unable to calculate that similarity

- But general calculation is based on the below

In [ ]:
### 5. Compare performances ###

import os


def jaccard_similarity(sentence1, sentence2):
    # Tokenize sentences into sets of words
    set1 = set(sentence1.lower().split())
    set2 = set(sentence2.lower().split())

    # Calculate Jaccard similarity
    intersection_size = len(set1.intersection(set2))
    union_size = len(set1.union(set2))

    # Avoid division by zero if both sets are empty
    similarity = intersection_size / union_size if union_size != 0 else 0.0

    return similarity


score_tesseract = 0
score_easyocr = 0
score_textract = 0
for image_path_ in os.listdir('/content/data'):
  image_path = os.path.join('/content/data', image_path_)

  gt = image_path[:-4].replace('_', ' ').lower()

  score_tesseract += jaccard_similarity(gt, read_text_tesseract(image_path).lower().replace('\n', '').replace('!', '').replace('?', '').replace('.', ''))
  score_easyocr += jaccard_similarity(gt, read_text_easyocr(image_path).lower().replace('\n', '').replace('!', '').replace('?', '').replace('.', ''))
  score_textract += jaccard_similarity(gt, read_text_textract(image_path).lower().replace('\n', '').replace('!', '').replace('?', '').replace('.', ''))

print('score tesseract:', score_tesseract / 100)
print('score_easyocr:', score_easyocr / 100)
print('score_textract:', score_textract / 100)